# 💬 Análisis de sentimiento en español con RNN / LSTM / GRU

**Objetivo:** clasificar textos en español como **positivo**, **neutral** o **negativo** con redes recurrentes (TensorFlow/Keras).

**Contenido:** 1) Dataset · 2) EDA · 3) Limpieza · 4) División train/val/test · 5) Vectorización · 6) Modelos y experimentos ·
7) Gráficas · 8) Evaluación · 9) Predicciones · 10) Guardado del modelo.

> 💡 Activa GPU (opcional): *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

## 0. Configuración

In [ ]:
import os, re, json, random
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid")
LABELS = ["positivo", "neutral", "negativo"]
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Selección y descripción del dataset

**multilingual-sentiments (config `spanish`)**, de Hugging Face (`tyqiangz/multilingual-sentiments`): textos en español
etiquetados con sentimiento **positivo, neutral o negativo** (licencia Apache 2.0).

- **Por qué es secuencial:** el sentimiento depende del orden y del contexto ("no está *mal*" ≠ "está *mal*").
- **Tipo de problema:** clasificación **multiclase** (3 clases) → salida `softmax` y `sparse_categorical_crossentropy`.
- Se unen los splits originales y se vuelve a dividir 70/15/15 de forma estratificada.

In [ ]:
BASE = "https://huggingface.co/datasets/tyqiangz/multilingual-sentiments/resolve/main/spanish/multilingual-sentiments-{}.parquet"
df = pd.concat([pd.read_parquet(BASE.format(s)) for s in ["train", "validation", "test"]], ignore_index=True)[["text", "label"]]
df["etiqueta"] = df["label"].map(dict(enumerate(LABELS)))
print("Forma:", df.shape)
# Verifica que el mapeo 0=positivo, 1=neutral, 2=negativo tenga sentido:
for lab in LABELS:
    print(f"\n--- {lab} ---")
    for t in df.loc[df.etiqueta == lab, "text"].head(3): print(" •", t[:120])

## 2. Análisis exploratorio de datos (EDA)

In [ ]:
print("Nulos:\n", df.isna().sum(), "\n")
print("Duplicados:", df.duplicated(subset="text").sum())
print(df["etiqueta"].value_counts(), "\n", df["etiqueta"].value_counts(normalize=True).round(3))
fig, ax = plt.subplots(figsize=(5, 3.5))
sns.countplot(data=df, x="etiqueta", order=LABELS, hue="etiqueta", legend=False, ax=ax)
ax.set_title("Distribución de clases"); plt.show()

In [ ]:
df["n_palabras"] = df["text"].str.split().str.len()
print(df.groupby("etiqueta")["n_palabras"].describe().round(1))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(data=df, x="n_palabras", hue="etiqueta", bins=40, element="step", ax=axes[0]); axes[0].set_title("Palabras por texto")
sns.boxplot(data=df, x="etiqueta", y="n_palabras", order=LABELS, hue="etiqueta", legend=False, ax=axes[1]); axes[1].set_title("Longitud por clase")
plt.tight_layout(); plt.show()
print("Percentil 95:", int(np.percentile(df["n_palabras"], 95)))

In [ ]:
# Stopwords SOLO para el EDA (el modelo las conserva: "no", "muy", "nada" cambian el sentimiento)
STOP = set("de la que el en y a los se del las un por con no una su para es al lo como más pero sus le ya o este si porque esta entre cuando muy sin sobre también me hasta hay donde desde todo nos todos uno les ni otros ese eso ante esto antes qué unos yo otro otra tanto esa estos mucho nada poco algo mi mis tu te ha han he era fue son ser está están estoy tiene tengo q lo".split())
def top_words(texts, n=12):
    c = Counter()
    for t in texts: c.update(w for w in re.findall(r"[a-záéíóúñü]+", t.lower()) if w not in STOP and len(w) > 2)
    return c.most_common(n)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, lab in zip(axes, LABELS):
    w, c = zip(*top_words(df.loc[df.etiqueta == lab, "text"]))
    ax.barh(w[::-1], c[::-1]); ax.set_title(f"Palabras frecuentes: {lab}")
plt.tight_layout(); plt.show()

## 3. Limpieza y preprocesamiento

- Eliminar duplicados y nulos.
- Minúsculas (se conservan tildes y ñ).
- URLs y @menciones → tokens genéricos; letras repetidas ("buenooooo" → "buenoo").
- **No** se eliminan stopwords. La puntuación (incluidos ¡ ¿) la elimina `TextVectorization`.

> `clean_text` es la misma función de `utils.py`, usada en `predict.py` y `app.py`.

In [ ]:
URL_RE, MENTION_RE, REPEAT_RE = re.compile(r"(https?://\S+|www\.\S+)"), re.compile(r"@\w+"), re.compile(r"(.)\1{2,}")
def clean_text(t):
    t = str(t).lower()
    t = URL_RE.sub(" urltoken ", t); t = MENTION_RE.sub(" usuario ", t); t = REPEAT_RE.sub(r"\1\1", t)
    return re.sub(r"\s+", " ", t).strip()

df = df.dropna().drop_duplicates(subset="text").reset_index(drop=True)
df["clean"] = df["text"].apply(clean_text)
print("Tras limpiar:", df.shape)
df[["text", "clean"]].sample(5, random_state=SEED)

## 4. División en entrenamiento, validación y prueba

70 / 15 / 15 **estratificada**, antes de construir el vocabulario (sin fuga de datos).

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
for n, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{n:5}: {len(d):5} |", d["etiqueta"].value_counts(normalize=True).round(3).to_dict())

## 5. Vectorización

`TextVectorization` se adapta **solo con entrenamiento** y va **dentro del modelo**, así el modelo guardado recibe texto.

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = int(np.percentile([len(t.split()) for t in train_df["clean"]], 95))
vectorizer = layers.TextVectorization(max_tokens=VOCAB_SIZE, output_mode="int", output_sequence_length=MAX_LEN,
                                      standardize="lower_and_strip_punctuation")
vectorizer.adapt(np.array(train_df["clean"], dtype=str))
print("MAX_LEN =", MAX_LEN, "| vocabulario =", vectorizer.vocabulary_size())
print("Primeras palabras:", vectorizer.get_vocabulary()[:15])
ej = train_df["clean"].iloc[0]; print("\nTexto :", ej); print("Tokens:", vectorizer(tf.constant([ej]))[0].numpy())

BATCH = 32
def make_ds(X, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((np.array(X, dtype=str), np.array(y, dtype="int32")))
    if shuffle: ds = ds.shuffle(len(X), seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = make_ds(train_df.clean, train_df.label, True), make_ds(val_df.clean, val_df.label), make_ds(test_df.clean, test_df.label)

## 6. Construcción y entrenamiento de la red neuronal

`TextVectorization → Embedding → [SimpleRNN | LSTM | GRU | BiLSTM] → Dropout → Dense(32) → Dropout → Dense(3, softmax)`

- **Pérdida:** `sparse_categorical_crossentropy` (etiquetas enteras, 3 clases).
- **Regularización:** Dropout 0.4 + `EarlyStopping` (patience=3) + `ModelCheckpoint`.
- **Desbalance:** `class_weight` balanceado.
- Se comparan 4 arquitecturas y se elige la mejor por **F1 macro en validación** (nunca en test).

In [ ]:
EMBED_DIM, UNITS, DROPOUT = 64, 64, 0.4
def build_model(kind):
    inp = keras.Input(shape=(), dtype=tf.string, name="texto")
    x = layers.Embedding(vectorizer.vocabulary_size(), EMBED_DIM, mask_zero=True)(vectorizer(inp))
    x = {"SimpleRNN": lambda: layers.SimpleRNN(UNITS), "LSTM": lambda: layers.LSTM(UNITS), "GRU": lambda: layers.GRU(UNITS),
         "BiLSTM": lambda: layers.Bidirectional(layers.LSTM(UNITS))}[kind]()(x)
    x = layers.Dropout(DROPOUT)(x); x = layers.Dense(32, activation="relu")(x); x = layers.Dropout(DROPOUT)(x)
    out = layers.Dense(len(LABELS), activation="softmax", name="probabilidades")(x)
    m = keras.Model(inp, out, name=f"sentimiento_{kind.lower()}")
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

build_model("BiLSTM").summary()
w = compute_class_weight("balanced", classes=np.arange(3), y=train_df["label"].values)
CLASS_WEIGHT = {i: float(v) for i, v in enumerate(w)}; print("class_weight:", CLASS_WEIGHT)

In [ ]:
os.makedirs("modelo", exist_ok=True)
modelos, historias, resumen = {}, {}, []
yv = val_df["label"].values
for nombre in ["SimpleRNN", "LSTM", "GRU", "BiLSTM"]:
    print(f"\n===== Entrenando {nombre} =====")
    tf.keras.utils.set_random_seed(SEED)
    m = build_model(nombre)
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
          keras.callbacks.ModelCheckpoint(f"modelo/ckpt_{nombre}.keras", monitor="val_loss", save_best_only=True)]
    h = m.fit(train_ds, validation_data=val_ds, epochs=25, class_weight=CLASS_WEIGHT, callbacks=cb, verbose=2)
    pv = m.predict(val_ds, verbose=0); yp = pv.argmax(1)
    modelos[nombre], historias[nombre] = m, h
    resumen.append({"modelo": nombre, "épocas": len(h.history["loss"]), "val_accuracy": accuracy_score(yv, yp),
                    "val_f1_macro": f1_score(yv, yp, average="macro"), "val_auc_ovr": roc_auc_score(yv, pv, multi_class="ovr")})
tabla = pd.DataFrame(resumen).set_index("modelo").round(4); tabla

## 7. Gráficas del entrenamiento

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey="row")
for j, (n, h) in enumerate(historias.items()):
    axes[0, j].plot(h.history["loss"], label="train"); axes[0, j].plot(h.history["val_loss"], label="val"); axes[0, j].set_title(f"{n} — pérdida")
    axes[1, j].plot(h.history["accuracy"], label="train"); axes[1, j].plot(h.history["val_accuracy"], label="val"); axes[1, j].set_title(f"{n} — accuracy")
    axes[1, j].set_xlabel("época")
axes[0, 0].legend(); axes[1, 0].legend(); plt.tight_layout(); plt.show()
mejor = tabla["val_f1_macro"].idxmax(); print(f"✅ Mejor modelo por F1 macro en validación: {mejor}")

Si la pérdida de validación sube mientras la de entrenamiento baja, hay sobreajuste; `EarlyStopping` restaura los mejores pesos. Compara la estabilidad de SimpleRNN frente a LSTM/GRU (el gradiente se desvanece más en secuencias largas).

## 8. Evaluación en el conjunto de prueba

In [ ]:
best = modelos[mejor]; y_test = test_df["label"].values
probs = best.predict(test_ds, verbose=0); preds = probs.argmax(1)
metrics = {"modelo": mejor, "accuracy": accuracy_score(y_test, preds),
           "precision_macro": precision_score(y_test, preds, average="macro"), "recall_macro": recall_score(y_test, preds, average="macro"),
           "f1_macro": f1_score(y_test, preds, average="macro"), "f1_weighted": f1_score(y_test, preds, average="weighted"),
           "roc_auc_ovr": roc_auc_score(y_test, probs, multi_class="ovr"), "confusion_matrix": confusion_matrix(y_test, preds).tolist(), "n_test": int(len(y_test))}
print(classification_report(y_test, preds, target_names=LABELS, digits=4)); print("ROC-AUC OvR:", round(metrics["roc_auc_ovr"], 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=LABELS, cmap="Blues", ax=axes[0]); axes[0].set_title(f"Matriz de confusión — {mejor}")
yb = label_binarize(y_test, classes=[0, 1, 2])
for i, lab in enumerate(LABELS):
    fpr, tpr, _ = roc_curve(yb[:, i], probs[:, i]); axes[1].plot(fpr, tpr, label=f"{lab} (AUC={roc_auc_score(yb[:, i], probs[:, i]):.3f})")
axes[1].plot([0, 1], [0, 1], "k--", alpha=.4); axes[1].legend(); axes[1].set_title("ROC uno-contra-resto"); plt.tight_layout(); plt.show()

**Interpretación:** en sentimiento, la clase *neutral* suele ser la más difícil (se confunde con las otras dos). Revisa la matriz de confusión y el recall por clase, no solo la accuracy.

## 9. Predicciones de prueba (≥ 10)

In [ ]:
res = test_df.copy(); res["predicción"] = [LABELS[i] for i in preds]; res["confianza"] = probs.max(1); res["correcta"] = res["predicción"] == res["etiqueta"]
muestra = pd.concat([g.sample(4, random_state=SEED) for _, g in res.groupby("etiqueta")])[["text", "etiqueta", "predicción", "confianza", "correcta"]].reset_index(drop=True)
muestra["text"] = muestra["text"].str.slice(0, 85); pd.set_option("display.max_colwidth", 95); muestra.round(3)

In [ ]:
err = res[~res["correcta"]][["text", "etiqueta", "predicción", "confianza"]].round(3)
print(f"Errores: {len(err)} de {len(res)}"); err.head(10)

## 10. Guardado del modelo

Se guarda en formato `.keras` con la capa `TextVectorization` incluida, y se verifica que al recargarlo da las mismas predicciones.

In [ ]:
os.makedirs("reports", exist_ok=True)
best.save("modelo/modelo_sentimiento.keras")
config = {"model": mejor.lower(), "max_len": MAX_LEN, "vocab_size": VOCAB_SIZE, "vocab_real": int(vectorizer.vocabulary_size()),
          "embed_dim": EMBED_DIM, "units": UNITS, "dropout": DROPOUT, "batch_size": BATCH, "seed": SEED, "labels": LABELS}
json.dump(config, open("modelo/config.json", "w"), indent=2); json.dump(metrics, open("reports/metrics.json", "w"), indent=2)
tabla.to_csv("reports/comparacion_modelos.csv"); muestra.to_csv("reports/predicciones_test.csv", index=False)

reloaded = keras.models.load_model("modelo/modelo_sentimiento.keras")
frases = ["Me encantó, llegó rápido y la calidad es excelente", "Es un producto normal, cumple lo básico", "Pésimo, llegó roto y nadie respondió mis reclamos"]
for f, p in zip(frases, reloaded.predict(tf.constant([clean_text(f) for f in frases]), verbose=0)):
    print(f"{LABELS[p.argmax()]:9} ({p.max():.2f}) | {f}")
print("\nDiferencia máx. original vs recargado:", float(np.abs(best.predict(test_ds, verbose=0) - reloaded.predict(test_ds, verbose=0)).max()))

In [ ]:
import shutil
for f in os.listdir("modelo"):
    if f.startswith("ckpt_"): os.remove(os.path.join("modelo", f))
shutil.make_archive("resultados_proyecto", "zip", ".", "modelo"); shutil.make_archive("reportes_proyecto", "zip", ".", "reports")
try:
    from google.colab import files
    files.download("resultados_proyecto.zip"); files.download("reportes_proyecto.zip")
except ImportError:
    print("Fuera de Colab: los archivos están en ./modelo y ./reports")

## 11. Conclusiones

- Indica qué arquitectura ganó y por cuánto (tabla de la sección 6).
- Analiza qué clase se confunde más (probablemente *neutral*) y los errores de la sección 9.
- Limitaciones: dataset de reseñas (puede rendir peor con otros dominios), ironía y sarcasmo.
- Mejoras: embeddings preentrenados en español, más datos, o un Transformer (BETO).

**Siguiente paso:** copia `modelo/` y `reports/` a la carpeta del proyecto y ejecuta `streamlit run app.py`.